In [1]:
!pip install pytabkit

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 364.0/364.0 kB 11.9 MB/s eta 0:00:00


In [2]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
import xgboost as xgb
import lightgbm as lgb
import catboost as cb
from sklearn.base import clone, BaseEstimator, TransformerMixin
from sklearn.impute import SimpleImputer
from sklearn.model_selection import KFold, StratifiedKFold
from sklearn.preprocessing import TargetEncoder, LabelEncoder
from sklearn.utils.validation import check_is_fitted
from pytabkit import RealMLP_TD_Classifier, TabM_D_Classifier
from itertools import combinations
from sklearn.metrics import roc_auc_score
from typing import List, Union, Optional
import copy
import warnings
warnings.filterwarnings('ignore')
# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/playground-series-s6e2/sample_submission.csv
/kaggle/input/playground-series-s6e2/train.csv
/kaggle/input/playground-series-s6e2/test.csv
/kaggle/input/heartdisease/Heart_Disease_Prediction.csv


In [3]:
class config:
    SEED = 42
    N_FOLDS = 5
    TARGET = 'Heart Disease'
    
    INPUT_DIR = '/kaggle/input/playground-series-s6e2'

class_mapping = {
    'Presence': 1,
    'Absence': 0
}
rev_class_mapping = {
    0: 'Absence',
    1: 'Presence'
}

CONFIG = config()

In [4]:
train = pd.read_csv(f'{CONFIG.INPUT_DIR}/train.csv')
train['source'] = 'train'
test = pd.read_csv(f'{CONFIG.INPUT_DIR}/test.csv')
test['source'] = 'test'
sample_sub = pd.read_csv(f'{CONFIG.INPUT_DIR}/sample_submission.csv')

org = pd.read_csv('/kaggle/input/heartdisease/Heart_Disease_Prediction.csv')
org['source'] = 'original'
# strat_feature

In [5]:
NUMS = [col for col in train.columns if col not in ['id', 'Heart Disease', 'source']]
# NUMS = [col for col in NUMS if col not in BINS]
HIGH_CARDINALITY = [col for col in NUMS if train[col].nunique() > 40]

In [6]:
# BINS = []
# for q in [5]:
#     for c in HIGH_CARDINALITY:
#         n = f'{c}_{q}_bin'
#         train_bins, bins = pd.qcut(train[c], q=q, labels=False, retbins=True, duplicates='drop')
#         train[n] = train_bins
#         test[n] = pd.cut(test[c], bins=bins, labels=False, include_lowest=True)
#         BINS.append(n)

# print(BINS)
# print('='*30)
# print(len(BINS))

In [7]:
combine = pd.concat([train.drop(columns='id'), test.drop(columns='id'), org], ignore_index=True).reset_index()

In [8]:
combine.columns

Index(['index', 'Age', 'Sex', 'Chest pain type', 'BP', 'Cholesterol',
       'FBS over 120', 'EKG results', 'Max HR', 'Exercise angina',
       'ST depression', 'Slope of ST', 'Number of vessels fluro', 'Thallium',
       'Heart Disease', 'source'],
      dtype='object')

In [9]:
# def feature_engineering(df):
#     BP_THRESHOLD = 140
#     CHOL_THRESHOLD = 240

#     df = df.copy()
    
#     df['elevated_bp'] = (df['BP'] >= BP_THRESHOLD).astype(int)
#     df['elevated_chol'] = (df['Cholesterol'] >= CHOL_THRESHOLD).astype(int)

#     # df['risk_factor_count'] = df['elevated_bp'] + df['elevated_chol']
#     # df['risk_age'] = df['Age'] * df['risk_factor_count']
#     return df

# combine = feature_engineering(combine)

In [10]:
for df, name in zip([train, test, org], ['train', 'test', 'original']):
    print(f'NULL VALUE COUNTS FOR {name}:')
    print(df.isnull().sum())
    print('='*30)
    print(f'{name} shape:')
    print(df.shape)
    print('='*30)
    if name == 'train':
        print('General EDA -- TRAIN ONLY', end='\n')
        print(f'Dtypes :', end='\n')
        print(df.dtypes)
        print('='*30)
        print(f'NUMBER OF UNIQUE VALUES :', end='\n')
        print(df.nunique())
        print('='*30)
    

NULL VALUE COUNTS FOR train:
id                         0
Age                        0
Sex                        0
Chest pain type            0
BP                         0
Cholesterol                0
FBS over 120               0
EKG results                0
Max HR                     0
Exercise angina            0
ST depression              0
Slope of ST                0
Number of vessels fluro    0
Thallium                   0
Heart Disease              0
source                     0
dtype: int64
train shape:
(630000, 16)
General EDA -- TRAIN ONLY
Dtypes :
id                           int64
Age                          int64
Sex                          int64
Chest pain type              int64
BP                           int64
Cholesterol                  int64
FBS over 120                 int64
EKG results                  int64
Max HR                       int64
Exercise angina              int64
ST depression              float64
Slope of ST                  int64
Number of ves

In [11]:
print(NUMS)
print(HIGH_CARDINALITY)

['Age', 'Sex', 'Chest pain type', 'BP', 'Cholesterol', 'FBS over 120', 'EKG results', 'Max HR', 'Exercise angina', 'ST depression', 'Slope of ST', 'Number of vessels fluro', 'Thallium']
['Age', 'BP', 'Cholesterol', 'Max HR', 'ST depression']


In [12]:
CATS = []
for c in NUMS:
    n = f'{c}_cat'
    combine[n] = combine[c].astype(str).astype('category')
    CATS.append(n)

print(CATS)
print('='*30)
print(len(CATS))

['Age_cat', 'Sex_cat', 'Chest pain type_cat', 'BP_cat', 'Cholesterol_cat', 'FBS over 120_cat', 'EKG results_cat', 'Max HR_cat', 'Exercise angina_cat', 'ST depression_cat', 'Slope of ST_cat', 'Number of vessels fluro_cat', 'Thallium_cat']
13


In [13]:
# INTER = []
# for c1, c2 in combinations(CATS, 2):
#     n = f'{c1}_{c2}'
#     combine[n] = (combine[c1].astype(str) + '_' + combine[c2].astype(str)).astype('category')
#     INTER.append(n)

# print(INTER)
# print('='*30)
# print(len(INTER))

In [14]:
# INTER1 = []
# for c1, c2 in combinations(NUMS, 2):
#     n = f'{c1}_{c2}'
#     combine[n] = combine[c1] * combine[c2]
#     INTER1.append(n)

# print(INTER1)
# print('='*30)
# print(len(INTER1))

In [15]:
# ENC = []

# for c in INTER:
#     n = f'{c}_enc'
#     combine[n], _ = pd.factorize(combine[c])
#     ENC.append(n)

# print(ENC)
# print('='*30)
# print(len(ENC))

In [16]:
train = combine.loc[combine['source']=='train']
test = combine.loc[combine['source']=='test']
org = combine.loc[combine['source']=='original']

In [17]:
# print(train.groupby('risk_factor_count')['Heart Disease'].mean())
# print(train.groupby('elevated_bp')['Heart Disease'].mean())

In [18]:
# org[CONFIG.TARGET] = org[CONFIG.TARGET].map(class_mapping)
# TE = []
# TE1 = []
# COND_TE = []

# # ALL_CATS = CATS + CATS1 + BINS
# GLOBAL_MEAN = org[config.TARGET].mean()
# # GLOBAL_STD = train_org_n[config.TARGET].std()
# GLOBAL_COUNT = org[config.TARGET].count()

# ALPHA = 10 

# for c in CATS:
#     # if i%5==0: print(i, end='===')
#     for target in [config.TARGET]:
#         # if target=='study_hours':
#         #     if c in CATS:
#         #         tmp_mean = train_org_n.groupby(c)[target].mean()
#         #         tmp_median = train_org_n.groupby(c)[target].median()
#         #         tmp_std = train_org_n.groupby(c)[target].std()
#         #         tmp_min = train_org_n.groupby(c)[target].min()
#         #         tmp_max = train_org_n.groupby(c)[target].max()

#         # else:
#         tmp_mean = org.groupby(c)[target].mean()
#         tmp_count = org.groupby(c)[target].count()
#             # tmp_median = train_org_n.groupby(c)[target].median()
#         tmp_std = org.groupby(c)[target].std()
#             # tmp_min = train_org_n.groupby(c)[target].min()
#             # tmp_max = train_org_n.groupby(c)[target].max()

        
#         # tmp_count = train_org_n.groupby(c)[target].size()
#         # tmp_mean_delta = tmp_mean - GLOBAL_MEAN
#         # tmp_std_delta = tmp_std - GLOBAL_STD
#         # tmp_mean_ratio = tmp_mean / GLOBAL_MEAN
#         # tmp_std_ratio = tmp_std / GLOBAL_STD
#         # tmp_count = train_org_n.groupby(c)[config.TARGET].size()

#         n_mean = f'TE_{c}_{target}_mean'
#         n_count = f'TE_{c}_{target}_count'
#         # n_median = f'TE_{c}_{target}_median'
#         n_std = f'TE_{c}_{target}_std'
#         # n_min = f'TE_{c}_{target}_min'
#         # n_max = f'TE_{c}_{target}_max'
#         # n_count = f'TE_{c}_{target}_count'
#         # n_mean_delta = f'TE_{c}_{target}_mean_delta'
#         # n_std_delta = f'TE_{c}_{target}_std_delta'
#         # n_mean_ratio = f'TE_{c}_{target}_mean_ratio'
#         # n_std_ratio = f'TE_{c}_{target}_std_ratio'

#         # n_c = f'TE_{c}_count'
#         print(f'{n_mean}, {n_count}, {n_std}', end=' ')
#         tmp_mean.name = n_mean
#         # tmp_median.name = n_median
#         tmp_std.name = n_std
#         # tmp_min.name = n_min
#         # tmp_max.name = n_max
#         tmp_count.name = n_count
#         # tmp_mean_delta.name = n_mean_delta
#         # tmp_std_delta.name = n_std_delta
#         # tmp_mean_ratio.name = n_mean_ratio
#         # tmp_std_ratio.name = n_std_ratio
        
    
#         stats = (pd.concat([tmp_mean, tmp_count, tmp_std], axis=1).reset_index().rename(columns={'index': c}))
#         org = org.merge(stats, on=c, how='left')
#         train = train.merge(stats, on=c, how='left')
#         test = test.merge(stats, on=c, how='left')

#         if target==config.TARGET:
#             TE.append(n_mean)
#             # TE.append(n_median)
#             TE.append(n_std)
#             # TE.append(n_min)
#             # TE.append(n_max)
#             TE.append(n_count)
#             # TE.append(n_mean_delta)
#             # TE.append(n_std_delta)
#             # TE.append(n_mean_ratio)
#             # TE.append(n_std_ratio)

#         else:
#             TE1.append(n_mean)
#             # TE.append(n_median)
#             # TE1.append(n_std)
#             # TE1.append(n_min)
#             # TE1.append(n_max)   
#     # TE.append(n_c)


In [19]:
for df in [org, train, test]:
    int_cols = df.select_dtypes(include=['int64']).columns.to_list()
    float_cols = df.select_dtypes(include=['float64']).columns.to_list()
    df[int_cols] = df[int_cols].astype('int32')
    df[float_cols] = df[float_cols].astype('float32')

In [20]:
def freq_encode(X, features, freq_encodings):
    """Frequency encoding for multiple features"""
    X_freq = X.copy()
    for c in features:
        X_freq[f'{c}_freq'] = X[c].map(freq_encodings[c]).astype('float32').fillna(0)
    return X_freq

def target_mean_encode(X, features, target_mean_dict, global_mean):
    """Target mean encoding for multiple features"""
    X_mean = X.copy()
    for c in features:
        X_mean[f'{c}_mean'] = X[c].map(target_mean_dict[c]).astype('float32').fillna(global_mean)
    return X_mean

def target_count_encode(X, features, target_count_dict, global_count):
    """Target count encoding for multiple features"""
    X_count = X.copy()
    for c in features:
        X_count[f'{c}_count'] = X[c].map(target_count_dict[c]).astype('float32').fillna(global_count)
    return X_count

def kbins_discretize(X, numeric_cols, n_bins=10):
    """Create binned versions of numeric features"""
    X_binned = X.copy()
    for col in numeric_cols:
        # Create bins
        bins = pd.qcut(X[col], q=n_bins, labels=False, duplicates='drop')
        X_binned[f'{col}_bin'] = bins.astype('category')
    return X_binned

In [21]:
FEATURES = [col for col in train.columns if col not in ['id', 'Heart Disease', 'source', 'index', 'strat_feature']]
# FEATURES = [col for col in FEATURES if col not in INTER]
print(FEATURES)
print(len(FEATURES))

X = train[FEATURES]
X_org = org[FEATURES]

y = train[CONFIG.TARGET].map(class_mapping)
y_org = org[CONFIG.TARGET].map(class_mapping)

X_test = test[FEATURES]

['Age', 'Sex', 'Chest pain type', 'BP', 'Cholesterol', 'FBS over 120', 'EKG results', 'Max HR', 'Exercise angina', 'ST depression', 'Slope of ST', 'Number of vessels fluro', 'Thallium', 'Age_cat', 'Sex_cat', 'Chest pain type_cat', 'BP_cat', 'Cholesterol_cat', 'FBS over 120_cat', 'EKG results_cat', 'Max HR_cat', 'Exercise angina_cat', 'ST depression_cat', 'Slope of ST_cat', 'Number of vessels fluro_cat', 'Thallium_cat']
26


In [22]:
xgb_params = {
    'n_estimators': 10000,
    'learning_rate': 0.005,
    'subsample': 0.8,
    # 'colsample_by_tree': 0.7,
    # 'sampling_method': 'gradient_based',
    # 'reg_alpha': 2.0,
    # 'reg_lambda': 4.0,
    'eval_metric': 'auc',
    'enable_categorical': True,
    'tree_method': 'hist',
    'device': 'cuda',
    'early_stopping_rounds': 100,
    'random_state': CONFIG.SEED
}

lgb_params = {"objective": "binary",
                "metric"               : "auc",
               'device'               : "cpu",
                'learning_rate'        : 0.005,
                'n_estimators'         : 3000 ,
                 'max_depth'            : 6,
                  'subsample'            : 0.70,
                 'colsample_bytree'     : 0.30,
                 'reg_lambda'           : 3.50,
                   'reg_alpha'            : 0.10,
                  'verbosity'            : -1,
                  'random_state'         : CONFIG.SEED,
                                              } 

cat_params = {
    'iterations': 10_000,          # same as n_estimators
    'learning_rate': 0.005,
    'depth': 8,                    # max_depth equivalent
    'subsample': 0.8,              # bagging
    'colsample_bylevel': 0.7,      # feature fraction per split
    'reg_lambda': 4.0,             # L2
    'random_seed': CONFIG.SEED,
    'early_stopping_rounds': 100,
    'eval_metric': 'AUC',
    'thread_count': -1,            # use all cores
    'verbose': 200,                # same as XGB verbose
    # 'verbosity':-1
    # 'cat_features': CATS,          # list of column names / indices
}

real_mlp_params = {
        'device': 'cuda',
        'random_state': 42,
        'verbosity': 2,
        'val_metric_name': '1-auc_ovo',
        'n_epochs': 60,
        'batch_size': 1024,
        'n_ens': 8,
        'use_early_stopping': True,
        'early_stopping_additive_patience': 20,
        'early_stopping_multiplicative_patience': 1,
        'act': "mish",
        'embedding_size': 8,
        'first_layer_lr_factor': 0.5962121993798933,
        'hidden_sizes': "rectangular",
        'hidden_width': 384,
        'lr': 0.01,
        'ls_eps': 0.0,
        'ls_eps_sched': "coslog4",
        'max_one_hot_cat_size': 18,
        'n_hidden_layers': 3,
        'p_drop': 0.1,
        'p_drop_sched': "flat_cos",
        'plr_hidden_1': 16,
        'plr_hidden_2': 8,
        'plr_lr_factor': 0.1151437622270563,
        'plr_sigma': 2.3316811282666916,
        'scale_lr_factor': 2.244801835541429,
        'sq_mom': 1.0 - 0.011834054955582318,
        'wd': 0.02369230879235962,
    }

In [23]:
X.dtypes

Age                               int32
Sex                               int32
Chest pain type                   int32
BP                                int32
Cholesterol                       int32
FBS over 120                      int32
EKG results                       int32
Max HR                            int32
Exercise angina                   int32
ST depression                   float32
Slope of ST                       int32
Number of vessels fluro           int32
Thallium                          int32
Age_cat                        category
Sex_cat                        category
Chest pain type_cat            category
BP_cat                         category
Cholesterol_cat                category
FBS over 120_cat               category
EKG results_cat                category
Max HR_cat                     category
Exercise angina_cat            category
ST depression_cat              category
Slope of ST_cat                category
Number of vessels fluro_cat    category


In [24]:
# X = X.iloc[:1000]
# y = y.iloc[:1000]
# X_test = X_test.iloc[:1000]

In [25]:
skf = StratifiedKFold(n_splits=CONFIG.N_FOLDS, random_state=CONFIG.SEED, shuffle=True)
kf = KFold(n_splits=CONFIG.N_FOLDS, random_state=CONFIG.SEED, shuffle=True)

oof_preds = np.zeros(len(X))
realmlp_oof_preds = np.zeros(len(X))
test_preds = np.zeros(len(X_test))
realmlp_test_preds = np.zeros(len(X_test))
strat_cols = ['Thallium', 'Chest pain type', 'Heart Disease']
le = LabelEncoder()
stratify_feature = le.fit_transform(train[strat_cols].astype(str).agg('_'.join, axis=1))
# stratify_feature = X['strat_feature']
pseudo_threshold = 0.5

for i, (train_idx, val_idx) in enumerate(skf.split(X, stratify_feature), 1):
    X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]
    X_test_n = X_test.copy()
    X_org_n = X_org.copy()
    y_org_n = y_org.copy()
    X_org_n = pd.concat([X_org_n]*5, axis=0, ignore_index=True)
    y_org_n = pd.concat([y_org_n]*5, axis=0, ignore_index=True)
    X_train = pd.concat([X_train, X_org_n], axis=0, ignore_index=True)
    y_train = pd.concat([y_train, y_org_n], axis=0, ignore_index=True)
    # model = clone(xgb.XGBClassifier(**xgb_params))
    # model = clone(cb.CatBoostClassifier(**cat_params))
    # model = clone(lgb.LGBMClassifier(**lgb_params))

    # model_realmlp = clone(RealMLP_TD_Classifier(**real_mlp_params))
    # freq_encodings = {}
    # for c in CATS:
    #     # n = f'{c}_freq'
    #     freq_encodings[c] = X_train[c].value_counts(normalize=True).to_dict()
    # X_train = freq_encode(X_train, CATS, freq_encodings)
    # X_val = freq_encode(X_val, CATS, freq_encodings)
    # X_test_n = freq_encode(X_test_n, CATS, freq_encodings)
        

    for c in CATS:
        n = f'{c}_mean_te'
        TE = TargetEncoder(cv=5, random_state=42, shuffle=True)
        X_train[c] = TE.fit_transform(pd.DataFrame(X_train[c]), y_train).flatten()
        X_val[c] = TE.transform(pd.DataFrame(X_val[c])).flatten()
        X_test_n[c] = TE.transform(pd.DataFrame(X_test[c])).flatten()

    # X_train_realmlp = X_train.copy()
    # X_val_realmlp = X_val.copy()
    # X_test_realmlp = X_test_n.copy()
    # for df in [X_train_realmlp, X_val_realmlp, X_test_realmlp]:
    #     for c in df.columns:
    #         df[c] = df[c].fillna(0)
    # model_realmlp.fit(X_train_realmlp, y_train,
    #               X_val_realmlp, y_val,
    #                  )
    # realmlp_val_preds = model_realmlp.predict_proba(X_val_realmlp)[:,1]
    # realmlp_oof_preds[val_idx] = realmlp_val_preds
    # realmlp_test_preds = model_realmlp.predict_proba(X_test_realmlp)[:,1]
    
    # realmlp_val_labels = (realmlp_val_preds > pseudo_threshold).astype(int)
    # realmlp_test_labels = (realmlp_test_preds > pseudo_threshold).astype(int)
    # for col in CATS:
    #     X_train[col] = X_train[col].astype(str)
    #     X_val[col] = X_val[col].astype(str)
    #     X_test_n[col] = X_test_n[col].astype(str)
    # for col in INTER:
    #     X_train[col] = X_train[col].astype(str)
    #     X_val[col] = X_val[col].astype(str)
    #     X_test_n[col] = X_test_n[col].astype(str)
        
    # TE = TargetEncoder(
    #     agg_funcs=['mean'], n_folds=5, smooth='auto', drop_original=False, cv_strategy='kfold', handle_unknown='nan', handle_missing='nan',
    #     verbose=1
    # )
    # X_train = TE.fit_transform(X_train, y_train, columns=CATS)
    # X_val = TE.transform(X_val)
    # X_test_n = TE.transform(X_test_n)

    # TE1 = TargetEncoder(
    #     agg_funcs=['mean'], n_folds=5, smooth='auto', drop_original=True, cv_strategy='kfold', handle_unknown='nan', handle_missing='nan',
    #     verbose=1
    # )
    # X_train = TE1.fit_transform(X_train, y_train, columns=INTER)
    # X_val = TE1.transform(X_val)
    # X_test_n = TE1.transform(X_test_n)

    # for col in CATS:
    #     X_train[col] = X_train[col].astype(str).astype('category')
    #     X_val[col] = X_val[col].astype(str).astype('category')
    #     X_test_n[col] = X_test_n[col].astype(str).astype('category')
    # for col in CATS:
    #     X_train[col] = X_train[col].astype(str).astype('category')
    #     X_val[col] = X_val[col].astype(str).astype('category')
    #     X_test_n[col] = X_test_n[col].astype(str).astype('category')
    # X_train.drop(columns=INTER, inplace=True)
    # X_val.drop(columns=INTER, inplace=True)
    # X_test_n.drop(columns=INTER, inplace=True)
    # print(X_train.dtypes)
    # print(f'SCORE FOR REALMLP FOLD{i} : {roc_auc_score(y_val, realmlp_val_preds)}')
    # X_train_xgb = pd.concat([X_train, X_val, X_test_n], axis=0, ignore_index=True)
    # y_train_xgb = pd.concat([y_train, pd.Series(realmlp_val_labels), pd.Series(realmlp_test_labels)], axis=0, ignore_index=True)
    print(X_train.shape)
    param_grid = {'colsample_bytree': 0.2364,
                  'gamma': 0.034283,
                  'max_depth': 6,
                  'reg_alpha': 0.71367,
                  'reg_lambda': 4.43564,
                  'subsample': 0.59394}

    model = xgb.XGBClassifier(**param_grid,
                          n_estimators=10000,
                          objective='binary:logistic',
                          eval_metric='auc',
                          learning_rate=0.01,
                          early_stopping_rounds=500,
                          max_bin=1024,
                          random_state=42,
                          enable_categorical=True,
                          device='cuda',
                          n_jobs=-1)
    # model = lgb.LGBMClassifier(**lgb_params)
    model.fit(X_train, y_train,
             eval_set=[(X_val, y_val)],
             verbose=500
             )
    preds = model.predict_proba(X_val)[:,1]
    oof_preds[val_idx] = preds
    test_preds += (model.predict_proba(X_test_n)[:, 1] / CONFIG.N_FOLDS)
    print(f'SCORE FOR FOLD{i} : {roc_auc_score(y_val, preds)}')
    
overall_roc_auc = roc_auc_score(y, oof_preds)
print(f'SCORE ACROSS ALL FOLDS : {overall_roc_auc}')

(505350, 26)
[0]	validation_0-auc:0.88474
[500]	validation_0-auc:0.95548
[1000]	validation_0-auc:0.95590
[1500]	validation_0-auc:0.95598
[2000]	validation_0-auc:0.95598
[2313]	validation_0-auc:0.95598
SCORE FOR FOLD1 : 0.9559870149400131
(505350, 26)
[0]	validation_0-auc:0.88352
[500]	validation_0-auc:0.95515
[1000]	validation_0-auc:0.95553
[1500]	validation_0-auc:0.95561
[2000]	validation_0-auc:0.95563
[2500]	validation_0-auc:0.95563
[2563]	validation_0-auc:0.95562
SCORE FOR FOLD2 : 0.9556327412795926
(505350, 26)
[0]	validation_0-auc:0.88351
[500]	validation_0-auc:0.95497
[1000]	validation_0-auc:0.95536
[1500]	validation_0-auc:0.95543
[2000]	validation_0-auc:0.95544
[2494]	validation_0-auc:0.95543
SCORE FOR FOLD3 : 0.955443300503374
(505350, 26)
[0]	validation_0-auc:0.88477
[500]	validation_0-auc:0.95530
[1000]	validation_0-auc:0.95573
[1500]	validation_0-auc:0.95581
[2000]	validation_0-auc:0.95582
[2416]	validation_0-auc:0.95581
SCORE FOR FOLD4 : 0.9558206675415668
(505350, 26)
[0]	

In [26]:
sample_sub['Heart Disease'] = test_preds
sample_sub.to_csv(f'submission_{overall_roc_auc}.csv', index=False)